# 中证800 V46 Anchor Parity Check

目的：先校准研究链路，避免继续把“代码差异”误判为“模型增强/模型退化”。

本 notebook 只做 anchor parity，不做任何新策略增强：

- label 固定：`alpha_1m`
- 参数固定：原始 V46 `BASE_PARAMS_FF10`
- 特征固定：原始 V46 `BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS`
- 特征筛选口径固定：仍用训练窗口前段 `diag_fit_df` 做相关性去重，模拟原始导出脚本
- 训练固定：全量 `train_df`，无 early stopping
- 导出固定轮数：60 / 80 / 94 / 100 / 120
- pkl 协议：兼容现有 `jq_backtest_v46_legacy_unsealed.py`

判定规则：如果这些固定轮数都不能接近旧 V46 anchor，先停止新实验，回头查数据、特征、训练窗口、fill values、LightGBM 版本和回测加载口径。

In [ ]:
import os
import gc
import pickle
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 240)

DATA_PATH = "train_csi800_factor_v40_data_enhancement.csv"
OUT_DIR = "csi800_ml_v46_anchor_parity_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_START = "2019-01-01"
TRAIN_END = "2025-03-31"
LABEL_END = "2025-03-31"
REQUIRE_LABEL_END_WITHIN_TRAIN = False

BENCHMARK = "000906.XSHG"
TARGET_COL = "alpha_1m"
TOP_N_CANDIDATES = 30
STOCK_NUM = 10
INDUSTRY_CAP_RATIO = 0.20
CORR_THRESHOLD = 0.70
DIAG_VALID_FRAC = 0.20
DIAG_VALID_MIN_MONTHS = 6
SEED = 42
FIXED_ITERS = [60, 80, 94, 100, 120]

# Optional: put the old strong pkl here if it exists in the same notebook runtime.
REFERENCE_PKL_PATH = "model_candidate_v46_lgb_direct_hybrid_l2_ff10_2019_2025q1_legacy_unsealed.pkl"

print("DATA_PATH =", DATA_PATH)
print("OUT_DIR =", OUT_DIR)
print("fixed iters =", FIXED_ITERS)


In [ ]:
BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio",
    "book_to_price_ratio",
    "earnings_yield",
    "sales_to_price_ratio",
    "cash_earnings_to_price_ratio",
    "earnings_to_price_ratio",
    "roe_ttm",
    "roa_ttm",
    "gross_profit_ttm",
    "operating_profit_to_total_profit",
    "net_operate_cash_flow_to_total_liability",
    "net_operating_cash_flow_coverage",
    "adjusted_profit_to_total_profit",
    "ACCA",
    "growth",
    "net_working_capital",
    "operating_profit_per_share",
    "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share",
    "super_quick_ratio",
    "MLEV",
    "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio",
    "momentum",
    "Rank1M",
    "sharpe_ratio_60",
    "Variance20",
    "liquidity",
    "beta",
    "ATR6",
    "MFI14",
    "DAVOL10",
    "VOL10",
    "VMACD",
    "VOSC",
    "Skewness20",
    "Kurtosis20",
]

HYBRID_LIGHT_EXTRA_COLS = [
    "liq_money_ratio_20_60",
    "liq_paused_count_20",
    "px_close_to_ma60",
    "px_drawdown_60",
    "ts_cash_flow_to_price_ratio_rank_mean_3m",
    "ts_Rank1M_rank_chg_1m",
]

CANDIDATE_COLS = BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS

BASE_PARAMS_FF10 = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 200,
    "feature_fraction": 1.0,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.3,
    "verbose": -1,
}

print("candidate cols:", len(CANDIDATE_COLS))


In [ ]:
def unique_keep_order(cols):
    seen = set()
    out = []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


def safe_rank_ic(a, b):
    s = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3 or s["a"].nunique() < 2 or s["b"].nunique() < 2:
        return np.nan
    return s["a"].rank(pct=True).corr(s["b"].rank(pct=True))


def build_corr_components(train_df, feature_cols, threshold):
    from collections import defaultdict
    corr = train_df[feature_cols].corr()
    graph = defaultdict(list)
    for i in range(len(feature_cols)):
        for j in range(i + 1, len(feature_cols)):
            v = corr.iloc[i, j]
            if not pd.isnull(v) and abs(v) > threshold:
                graph[feature_cols[i]].append(feature_cols[j])
                graph[feature_cols[j]].append(feature_cols[i])
    for col in feature_cols:
        graph[col]

    visited = set()
    comps = []
    def dfs(x, comp):
        visited.add(x)
        comp.append(x)
        for y in graph[x]:
            if y not in visited:
                dfs(y, comp)

    for col in feature_cols:
        if col not in visited:
            comp = []
            dfs(col, comp)
            comps.append(comp)
    return comps


def select_features_train_only(train_df, candidate_cols):
    cols = unique_keep_order([c for c in candidate_cols if c in train_df.columns])
    missing = train_df[cols].isnull().sum().to_dict()
    keep = []
    remove = []
    for comp in build_corr_components(train_df, cols, CORR_THRESHOLD):
        if len(comp) == 1:
            keep.append(comp[0])
        else:
            comp = sorted(comp, key=lambda x: (missing[x], x))
            keep.append(comp[0])
            remove.extend(comp[1:])
    return keep, remove


def split_diag_valid(train_df):
    months = sorted(pd.to_datetime(train_df["rebalance_date"].dropna().unique()))
    n_valid = max(DIAG_VALID_MIN_MONTHS, int(round(len(months) * DIAG_VALID_FRAC)))
    valid_months = set(months[-min(n_valid, max(1, len(months) - 1)):])
    fit = train_df[~train_df["rebalance_date"].isin(valid_months)].copy()
    valid = train_df[train_df["rebalance_date"].isin(valid_months)].copy()
    if fit.empty or valid.empty:
        fit, valid = train_df.copy(), train_df.copy()
    return fit, valid


def prepare_xy(df, feature_cols, target_col, fill_values=None):
    d = df.dropna(subset=[target_col]).copy()
    X = d[feature_cols].replace([np.inf, -np.inf], np.nan)
    y = d[target_col].astype(float)
    if fill_values is None:
        fill_values = X.median().replace([np.inf, -np.inf], np.nan).fillna(0)
    X = X.fillna(fill_values).fillna(0)
    return X, y, fill_values, d.index


In [ ]:
def load_train_df(path):
    if not os.path.exists(path):
        raise IOError("DATA_PATH not found: " + path)
    df = pd.read_csv(path)
    if "code" in df.columns and "stock" not in df.columns:
        df = df.rename(columns={"code": "stock"})
    for col in ["rebalance_date", "feature_date", "next_date"]:
        if col not in df.columns:
            raise ValueError("missing date column: " + col)
        df[col] = pd.to_datetime(df[col]).dt.normalize()
    if TARGET_COL not in df.columns:
        raise ValueError("missing target column: " + TARGET_COL)
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df = df.dropna(subset=["stock", "rebalance_date", "next_date", TARGET_COL]).copy()

    train = df[(df["rebalance_date"] >= pd.Timestamp(TRAIN_START)) & (df["rebalance_date"] <= pd.Timestamp(TRAIN_END))].copy()
    if REQUIRE_LABEL_END_WITHIN_TRAIN:
        train = train[train["next_date"] <= pd.Timestamp(LABEL_END)].copy()
    if train.empty:
        raise ValueError("train_df is empty")
    return df, train


df_all, train_df = load_train_df(DATA_PATH)
diag_fit_df, diag_valid_df = split_diag_valid(train_df)
feature_cols, removed_cols = select_features_train_only(diag_fit_df, CANDIDATE_COLS)

print("all shape:", df_all.shape)
print(df_all[["rebalance_date", "feature_date", "next_date"]].agg(["min", "max"]))
print("train shape:", train_df.shape, "months:", train_df["rebalance_date"].nunique())
print("train date:", train_df["rebalance_date"].min(), train_df["rebalance_date"].max())
print("diag_fit:", diag_fit_df.shape, "diag_valid:", diag_valid_df.shape)
print("features:", len(feature_cols), "removed:", len(removed_cols))
print(feature_cols)
print("removed:", removed_cols)
print(train_df[[TARGET_COL]].describe())


In [ ]:
def train_fixed_iter_model(fixed_iter):
    params = dict(BASE_PARAMS_FF10)
    params["seed"] = SEED

    X_train, y_train, fill_values, _ = prepare_xy(train_df, feature_cols, TARGET_COL)
    model = lgb.train(
        params,
        lgb.Dataset(X_train, label=y_train),
        num_boost_round=max(1, int(fixed_iter)),
    )

    train_pred = np.asarray(model.predict(X_train[feature_cols], num_iteration=fixed_iter)).reshape(-1)
    train_rank_ic = safe_rank_ic(y_train, train_pred)

    X_valid, y_valid, _, _ = prepare_xy(diag_valid_df, feature_cols, TARGET_COL, fill_values)
    valid_pred = np.asarray(model.predict(X_valid[feature_cols], num_iteration=fixed_iter)).reshape(-1)
    diag_rank_ic = safe_rank_ic(y_valid, valid_pred)

    return {
        "model": model,
        "fill_values": fill_values,
        "train_rows": int(len(X_train)),
        "train_rank_ic": train_rank_ic,
        "diag_rank_ic": diag_rank_ic,
    }


TRAINED = {}
for fixed_iter in FIXED_ITERS:
    print("training fixed_iter:", fixed_iter)
    TRAINED[int(fixed_iter)] = train_fixed_iter_model(int(fixed_iter))
    row = TRAINED[int(fixed_iter)]
    print("  train_rank_ic:", row["train_rank_ic"], "diag_rank_ic:", row["diag_rank_ic"])
    gc.collect()


In [ ]:
def make_research_version(fixed_iter):
    return "candidate_v46_anchor_parity_fixed_iter{}_alpha1m_20190101_20250331_legacy_unsealed".format(int(fixed_iter))


export_rows = []
for fixed_iter in FIXED_ITERS:
    trained = TRAINED[int(fixed_iter)]
    research_version = make_research_version(fixed_iter)
    model_file = "model_{}.pkl".format(research_version)
    out_path = os.path.join(OUT_DIR, model_file)

    bundle = {
        "objective": "v210_refit_fixed_iter_overlay",
        "research_version": research_version,
        "benchmark": BENCHMARK,
        "train_start": TRAIN_START,
        "train_end": TRAIN_END,
        "label_end": LABEL_END,
        "require_label_end_within_train": bool(REQUIRE_LABEL_END_WITHIN_TRAIN),
        "target_col": TARGET_COL,
        "target_note": "V46 anchor parity: original alpha_1m label, original V46 params/features, fixed iter only",
        "data_file": DATA_PATH,
        "protocol": "v46_anchor_parity_fixed_iter_full_train",
        "training_policy": "expanding",
        "param_set": "v46_base_ff10_original",
        "final_role": "v46_anchor_parity_check",
        "base_params": dict(BASE_PARAMS_FF10),
        "base_model": trained["model"],
        "base_feature_cols": list(feature_cols),
        "base_fill_values": dict(trained["fill_values"]),
        "base_best_iter": int(fixed_iter),
        "es_best_iter": np.nan,
        "model_iter": int(fixed_iter),
        "fixed_iter": int(fixed_iter),
        "base_inner_metrics": {
            "train_rank_ic": float(trained["train_rank_ic"]) if not pd.isnull(trained["train_rank_ic"]) else np.nan,
            "diag_rank_ic": float(trained["diag_rank_ic"]) if not pd.isnull(trained["diag_rank_ic"]) else np.nan,
        },
        "base_removed_features": list(removed_cols),
        "residual_model": None,
        "residual_feature_cols": [],
        "residual_fill_values": {},
        "overlay_weight": 0.0,
        "overlay_mode": "direct",
        "top_n_candidates": TOP_N_CANDIDATES,
        "stock_num": STOCK_NUM,
        "industry_cap_ratio": INDUSTRY_CAP_RATIO,
        "requires_v4_feature_adapter": True,
        "requires_industry_relative_adapter": False,
        "uses_time_weight": False,
        "uses_sample_weight": False,
        "uses_current_valid_for_training": False,
    }

    with open(out_path, "wb") as f:
        pickle.dump(bundle, f, protocol=2)

    export_rows.append({
        "fixed_iter": int(fixed_iter),
        "research_version": research_version,
        "model_file": model_file,
        "model_path": out_path,
        "train_rows": trained["train_rows"],
        "train_rank_ic": trained["train_rank_ic"],
        "diag_rank_ic": trained["diag_rank_ic"],
        "feature_count": len(feature_cols),
        "removed_feature_count": len(removed_cols),
        "train_start": TRAIN_START,
        "train_end": TRAIN_END,
        "label_end": LABEL_END,
        "require_label_end_within_train": bool(REQUIRE_LABEL_END_WITHIN_TRAIN),
    })

export_manifest_df = pd.DataFrame(export_rows)
manifest_path = os.path.join(OUT_DIR, "v46_anchor_parity_export_manifest.csv")
export_manifest_df.to_csv(manifest_path, index=False)
print(export_manifest_df)
print("manifest:", manifest_path)
print("saved files:")
for row in export_rows:
    print("  " + row["model_path"])


In [ ]:
# Optional reference metadata comparison. This is only a diagnostic helper.
# It does not affect exported parity models.
def summarize_bundle(bundle):
    return {
        "research_version": bundle.get("research_version"),
        "target_col": bundle.get("target_col"),
        "train_start": bundle.get("train_start"),
        "train_end": bundle.get("train_end"),
        "label_end": bundle.get("label_end"),
        "require_label_end_within_train": bundle.get("require_label_end_within_train"),
        "protocol": bundle.get("protocol"),
        "iteration_policy": bundle.get("iteration_policy"),
        "model_iter": bundle.get("model_iter"),
        "base_best_iter": bundle.get("base_best_iter"),
        "es_best_iter": bundle.get("es_best_iter"),
        "overlay_mode": bundle.get("overlay_mode"),
        "top_n_candidates": bundle.get("top_n_candidates"),
        "stock_num": bundle.get("stock_num"),
        "industry_cap_ratio": bundle.get("industry_cap_ratio"),
        "requires_v4_feature_adapter": bundle.get("requires_v4_feature_adapter"),
        "feature_count": len(bundle.get("base_feature_cols", [])),
        "base_params": bundle.get("base_params"),
    }

if os.path.exists(REFERENCE_PKL_PATH):
    ref = pickle.load(open(REFERENCE_PKL_PATH, "rb"))
    ref_summary = summarize_bundle(ref)
    cur_path = export_rows[0]["model_path"]
    cur = pickle.load(open(cur_path, "rb"))
    cur_summary = summarize_bundle(cur)
    compare_df = pd.DataFrame([ref_summary, cur_summary], index=["reference", "parity_first"])
    print(compare_df.T)

    ref_features = list(ref.get("base_feature_cols", []))
    cur_features = list(cur.get("base_feature_cols", []))
    print("same feature list:", ref_features == cur_features)
    if ref_features != cur_features:
        print("only in reference:", [x for x in ref_features if x not in cur_features])
        print("only in parity:", [x for x in cur_features if x not in ref_features])
else:
    print("reference pkl not found, skip metadata comparison:", REFERENCE_PKL_PATH)


In [ ]:
required = [
    "objective", "base_model", "base_feature_cols", "base_fill_values",
    "residual_feature_cols", "residual_fill_values", "overlay_weight", "overlay_mode",
]
for row in export_rows:
    loaded = pickle.load(open(row["model_path"], "rb"))
    missing = [k for k in required if k not in loaded]
    print("fixed_iter", row["fixed_iter"], "missing keys:", missing)
    print("  objective:", loaded["objective"], "mode:", loaded["overlay_mode"], "features:", len(loaded["base_feature_cols"]), "iter:", loaded["fixed_iter"])


## 回测记录区

逐个上传 pkl，用同一个 V46 回测文件只改 `g.model_file`。

| fixed_iter | 年化 | 超额 | 最大回撤 | 胜率 | 备注 |
|---:|---:|---:|---:|---:|---|
| 60 | | | | | |
| 80 | | | | | |
| 94 | | | | | |
| 100 | | | | | |
| 120 | | | | | |

结论：

- 是否存在接近旧 V46 anchor 的 fixed_iter：
- 如果没有，下一步排查项：数据文件 / 特征列表 / fill_values / LightGBM 版本 / 回测文件加载口径。
